# Notebook 07 - Market Basket Analysis

**Input:** `data/interim/transactions_clean.parquet` (1.04M rows, ~39K invoices, ~4,900 products)

**Output:**
- `data/processed/product_rules.parquet` - association rules with support, confidence, lift
- `data/processed/product_cross_sell.parquet` - top cross-sell recommendations per product
- `reports/figures/basket_*.png` - visualizations

## Why pivot from customer-level to product-level

Notebook 04-06 built the **strategy matrix** - who to target, when, and with what urgency. But that matrix doesn't answer one critical question: **what should we actually offer each customer?

Market basket analysis answers that. It finds products that are frequently purchased together, producing rules like:

> *Customers who buy the white t-light holder are 4x more likely to also buy the regency cakestand than a random shopper.*

Combined with the strategy matrix:
- Customer A is in "Urgent win-back" -> look at what they bought historically -> recommend the highest-lift complementary product -> personalized offer.

That's a complete recommendation: the right customer, the right product, the right time.

## The three core metrics

All of association rules mining boils down to 3 numbers:
1. **Support** = % of all invoices that contain item X (or both X and Y). Tells you how the common a pattern is.
2. **Confidence** = `P(X | Y)` = if someone buys X, what fraction also buy Y? Tells you how reliable the rule is.
3. **Lift** = `confidence / P(Y)` = how much MORE likely is Y given X, vs. a random shopper? **The single most important metric.** <br>
Lift > 1 = real association. Lift = 1 = independence (no signal). Lift < 1 = negative association.

Lift is what we'll rank by because confidence alone is misleading - a hugely popular product will have high confidence with everything else just because it's in everyrone's basket.

## Notebook structure:
1. Load and prepare transaction-level data
2. Filter products & invoices to a tracable subset
3. Build basket matrix (one-hot encoded baskets)
4. Run Apriori to find frequent itemsets
5. Generate association rule with support, confidence, lift
6. Inspect top rules
7. Build a per-product cross-sell lookup
8. Visualize the recommendation network
9. Save outputs for the recommendation engine

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.3f}'.format)
sns.set_style('whitegrid')

INTERIM_DIR = Path('../data/interim')
PROCESSED_DIR = Path('../data/processed')
REPORTS_DIR = Path('../reports/figures')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load transactions
We use `transactions_clean.parquet` (NOT the customer-level one). because market basket analysis is invoice-driven - guest checkouts contribute valid co-purchase signal even without a customer ID

In [5]:
df = pd.read_parquet(INTERIM_DIR / 'transactions_clean.parquet')
print(f'Loaded {len(df):,} rows across {df["Invoice"].nunique():,} invoices and {df["StockCode"].nunique():,} products')

Loaded 1,037,012 rows across 39,519 invoices and 4,897 products


## 2. Why filtering matters

**The comnbinatorial explosion problem:** with 4,899 unique products, there are 4899 x 4898 / 2 ≈ **12 million possible pairs,** and exponentially more triples and beyond. Apriori would take forever and memory on the full set.

**The solution:** keep only products that appear often enough to produce statistically reliable rules, and only invoices with enough items to actually form a basket. This inn't cheating - it's focusing the analysis on the products where reliable patterns can exist.

**Filters we'll apply:**
- Drop invoices with fewer than 2 distinct products (no basket = nothing to learn)
- Keep only products that appear in at least 1% of invoices (~396 invoices). Rare products = unreliable patterns.
- Cap basket size at the 95th percentile to prevent one huge wholesale order from skewing everything.

In [15]:
# Compute invoice-level basket sizes (unique products per invoice)
basket_sizes = df.groupby('Invoice')['StockCode'].nunique()

# Cap at 95th percentile to avoid one mega-order dominating
cap = int(basket_sizes.quantile(.95))
print(f'Basket size 95th percentile: {cap}')

valid_invoices = basket_sizes[(basket_sizes >= 2) & (basket_sizes <= cap)].index
print(f'Invoices kept (2 ≤ basket_size ≤ {cap}): {len(valid_invoices):,} of {df["Invoice"].nunique():,}')

# Filter
df_basket = df[df['Invoice'].isin(valid_invoices)].copy()

# Keep only products that appear in >= 1% of remaining invoices
min_invoice_count = max(int(0.01 * len(valid_invoices)), 50)
product_invoice_count = df_basket.groupby('StockCode')['Invoice'].nunique()
popular_products = product_invoice_count[product_invoice_count > min_invoice_count].index
print(f'\nProducts kept (≥{min_invoice_count} invoices): {len(popular_products):,} of {df["StockCode"].nunique():,}')

df_basket = df_basket[df_basket['StockCode'].isin(popular_products)].copy()
print(f'\nFiltered dataset: {len(df_basket):,} rows, {df_basket["Invoice"].nunique():,} invoices, {df_basket["StockCode"].nunique():,} products')

Basket size 95th percentile: 73
Invoices kept (2 ≤ basket_size ≤ 73): 34,374 of 39,519

Products kept (≥343 invoices): 527 of 4,897

Filtered dataset: 387,529 rows, 33,032 invoices, 527 products
